In [1]:
from pathlib import Path
from typing import Dict, List, Tuple
from docplex.mp.dvar import Var
from docplex.mp.model import Model


def ler_instancia_pd(
    caminho: Path,
) -> Tuple[int, List[str], List[str], List[List[float]]]:
    linhas = [
        linha.strip()
        for linha in caminho.read_text(encoding="utf-8").splitlines()
        if linha.strip()
    ]
    n = int(linhas[0])
    pessoas = linhas[1].split()
    tarefas = linhas[2].split()
    custos = [[float(v) for v in linha.split()] for linha in linhas[3 : 3 + n]]
    return n, pessoas, tarefas, custos


n, pessoas, tarefas, custos = ler_instancia_pd(Path("in.txt"))
print(f"Instância carregada: {n} pessoas ({pessoas}) e {n} tarefas ({tarefas}).")

Instância carregada: 3 pessoas (['1', '2', '3']) e 3 tarefas (['A', 'B', 'C']).


In [2]:
mdl = Model(name="Problema_da_Designacao")

# Variáveis binárias x_ij = 1 se a pessoa i for designada para a tarefa j
x: Dict[Tuple[int, int], Var] = {
    (i, j): mdl.binary_var(name=f"x_{pessoas[i]}_{tarefas[j]}")
    for i in range(n)
    for j in range(n)
}

# (1) Função Objetivo: minimizar o custo total de designação
mdl.minimize(
    mdl.sum(custos[i][j] * x[(i, j)] for i in range(n) for j in range(n))
)

# (2) Cada tarefa j deve ter exatamente uma pessoa designada
for j in range(n):
    mdl.add_constraint(
        mdl.sum(x[(i, j)] for i in range(n)) == 1,
        ctname=f"tarefa_{tarefas[j]}",
    )

# (3) Cada pessoa i deve ser designada a exatamente uma tarefa
for i in range(n):
    mdl.add_constraint(
        mdl.sum(x[(i, j)] for j in range(n)) == 1,
        ctname=f"pessoa_{pessoas[i]}",
    )

print("Modelo PD construído com sucesso.")

Modelo PD construído com sucesso.


In [3]:
sol = mdl.solve(log_output=False)

if sol is None:
    print("Nenhuma solução encontrada.")
else:
    print(f"Status: {mdl.solve_details.status}")
    print(f"Função Objetivo (Custo Mínimo): {sol.objective_value:.2f}\n")
    print("Designações (Pessoa -> Tarefa):")
    for i in range(n):
        for j in range(n):
            val = int(round(sol.get_value(x[(i, j)])))
            print(f"  x[{pessoas[i]}->{tarefas[j]}] = {val}   (custo: {custos[i][j]:.0f})")

Status: integer optimal solution
Função Objetivo (Custo Mínimo): 96.00

Designações (Pessoa -> Tarefa):
  x[1->A] = 0   (custo: 40)
  x[1->B] = 0   (custo: 37)
  x[1->C] = 1   (custo: 35)
  x[2->A] = 1   (custo: 36)
  x[2->B] = 0   (custo: 38)
  x[2->C] = 0   (custo: 34)
  x[3->A] = 0   (custo: 29)
  x[3->B] = 1   (custo: 25)
  x[3->C] = 0   (custo: 26)
